# 13 — Replay visualization: trajectory, actions, rewards, prompt/completion

Эта тетрадка открывает сохранённый replay artifact и показывает, что именно произошло в эпизоде: actions, rewards, terminal/fallback flags, prompt/completion и observation image, если она сохранена как массив. Для интерактивного покадрового просмотра используется тот же backend, что и `scripts/visualize_trajectory.py`.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Any
import numpy as np
# from scripts.visualize_trajectory import load_trajectory, normalize_image

def load_trajectory(path: Path) -> dict[str, Any]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict) or "steps" not in payload:
        raise ValueError(f"Trajectory file {path} is not a replay artifact")
    if not isinstance(payload["steps"], list):
        raise ValueError(f"Trajectory file {path} contains invalid steps")
    return payload


def normalize_image(obs: Any) -> np.ndarray | None:
    try:
        array = np.asarray(obs)
    except Exception:
        return None
    if array.size == 0:
        return None
    if array.ndim == 1:
        return None
    if array.ndim == 2:
        image = array
        image = image.astype(np.float32)
        image = image - image.min()
        if image.max() > 0:
            image = image / image.max() * 255.0
        image = np.stack([image, image, image], axis=-1)
    elif array.ndim == 3 and array.shape[2] in {1, 3, 4}:
        image = array.astype(np.float32)
        if image.dtype != np.uint8:
            image = image - image.min()
            if image.max() > 0:
                image = image / image.max() * 255.0
            image = image.astype(np.uint8)
        if image.shape[2] == 1:
            image = np.concatenate([image, image, image], axis=-1)
        elif image.shape[2] == 4:
            image = image[:, :, :3]
    else:
        return None
    if image.dtype != np.uint8:
        image = np.clip(image, 0, 255).astype(np.uint8)
    return image



In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook inside tunix-craftext.')

CANDIDATES = [
    ROOT / 'artifacts/trajectories/qwen-craftext-full-notebook/env-0.json',
    ROOT / 'artifacts/trajectories/qwen-craftext-latest.json',
    ROOT / 'examples/notebooks/artifacts/trajectories/caged-random-policy.json',
]
trajectory_path = next((path for path in CANDIDATES if path.is_file()), None)
if trajectory_path is None:
    raise FileNotFoundError(
        'No replay artifact found. Run notebook 05, 07, 09, 11 or 12 first, '
        'or set trajectory_path manually to a replay JSON.'
    )
payload = load_trajectory(trajectory_path)
steps = payload['steps']
print('trajectory:', trajectory_path)
print('schema:', payload.get('schema'))
print('backend:', payload.get('backend'))
print('steps:', len(steps))


In [ ]:
summary = []
for index, step in enumerate(steps):
    summary.append({
        't': index,
        'action': step.get('action_label'),
        'action_id': step.get('action_id'),
        'reward': step.get('reward'),
        'terminated': step.get('terminated'),
        'truncated': step.get('truncated'),
        'fallback': step.get('fallback_used'),
        'masked': step.get('masked_action'),
        'tokens': len(step.get('token_ids') or ()),
    })
summary


In [ ]:
xs = [row['t'] for row in summary]
rewards = [float(row['reward'] or 0.0) for row in summary]
actions = [str(row['action']) for row in summary]
fig, ax1 = plt.subplots(figsize=(10, 3))
ax1.plot(xs, rewards, marker='o', label='reward')
ax1.axhline(0, color='black', linewidth=0.8, alpha=0.4)
ax1.set_xlabel('step')
ax1.set_ylabel('reward')
ax1.set_title('Replay reward timeline')
for x, reward, action in zip(xs, rewards, actions):
    ax1.annotate(action, (x, reward), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
ax1.grid(True, alpha=0.25)
plt.show()


In [ ]:
step_index = 0  # change this and re-run the cell to inspect another replay step
step = steps[step_index]
print('step:', step_index)
print('action:', step.get('action_label'), 'reward:', step.get('reward'), 'terminated:', step.get('terminated'))
print('fallback:', step.get('fallback_used'), 'masked:', step.get('masked_action'))
print('raw completion:', step.get('raw_completion'))
print('prompt excerpt:')
print(str(step.get('prompt', ''))[:1200])
image = normalize_image(step.get('observation'))
if image is not None:
    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f'observation at step {step_index}')
    plt.show()
else:
    print('No renderable observation image in this replay step.')


## Интерактивный pygame viewer

Если нужен покадровый viewer с клавишами ←/→, запусти из корня репозитория:

```bash
uv run python scripts/visualize_trajectory.py \
  --trajectory artifacts/trajectories/qwen-craftext-full-notebook/env-0.json
```

Controls: arrows — prev/next, Home/End — first/last, Q/Esc — quit.
